<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module340/Lab4.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 4 — Four-Qubit H₂ Encoding and Jordan–Wigner Excitations
**Quantum Optimization and Simulation — VQE Laboratory Series**

Use the standard spin-orbital occupation model before reducing to two qubits.

**Suggested use:** 10–15 minute instructor demonstration followed by approximately one hour of independent work.

**Notebook style:** Most code is supplied. Complete the small items marked **YOUR TURN** and answer the reflection questions.

> Qiskit displays measured bitstrings as `q_(n-1)...q_0`. When orbital labels are written in the order `q0, q1, ...`, this notebook explicitly notes the convention.

## Learning objectives
- Use the standard four-spin-orbital representation of H₂.
- Prepare the Hartree–Fock occupation state.
- Enumerate valid two-electron configurations.
- See how Jordan–Wigner translates excitation operators into Pauli strings.

In [ ]:
# Run once in a fresh Google Colab session.
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-algorithms~=0.4" "qiskit-nature~=0.8"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer import AerSimulator

SEED = 123
SHOTS = 4096

def run_counts(qc, shots=SHOTS, noise_model=None):
    backend = AerSimulator(noise_model=noise_model)
    tqc = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=shots, seed_simulator=SEED).result()
    return result.get_counts()

def q0_first(qiskit_bits):
    return qiskit_bits.replace(" ", "")[::-1]

## Four spin orbitals

| Qubit | Spin orbital |
|---|---|
| \(q_0\) | bonding, spin up |
| \(q_1\) | bonding, spin down |
| \(q_2\) | antibonding, spin up |
| \(q_3\) | antibonding, spin down |

We use `1 = occupied`, `0 = empty`.

## Part A — Prepare Hartree–Fock

In [ ]:
hf = QuantumCircuit(4)
hf.x(0)
hf.x(1)

display(hf.draw("mpl"))
print(Statevector.from_instruction(hf))

In the logical order \(q_0q_1q_2q_3\), this is `1100`. Qiskit's statevector label is displayed in reverse qubit order.

## Part B — Enumerate all two-electron occupation configurations

In [ ]:
from itertools import product

configurations = [
    bits for bits in product([0,1], repeat=4)
    if sum(bits) == 2
]

print("Number of two-electron configurations:", len(configurations))
for bits in configurations:
    print("".join(map(str, bits)))

**Expected:** \(inom{4}{2}=6\) configurations.

## Part C — Jordan–Wigner mapping of a single excitation

In [ ]:
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper

# Excitation 0 -> 2 plus its Hermitian conjugate.
hopping = FermionicOp(
    {
        "+_2 -_0": 1.0,
        "+_0 -_2": 1.0,
    },
    num_spin_orbitals=4,
)

jw_hopping = JordanWignerMapper().map(hopping)
print(jw_hopping)

The mapped operator contains \(X/Y\) actions at the two endpoint modes and a parity \(Z\)-string between them. The exact signs depend on whether one maps a Hermitian hopping sum or the anti-Hermitian UCC generator.

## Part D — Double excitation

In [ ]:
double_generator = FermionicOp(
    {
        "+_2 +_3 -_1 -_0": 1.0,
        "+_0 +_1 -_3 -_2": -1.0,
    },
    num_spin_orbitals=4,
)

jw_double = JordanWignerMapper().map(double_generator)
print(jw_double)

### YOUR TURN
Count the Pauli strings in `jw_double` and identify how many contain one \(Y\) and how many contain three \(Y\)'s.

In [ ]:
labels = jw_double.paulis.to_labels()
print(labels)

# TODO: calculate these two counts.
one_y = 0
three_y = 0

print("one Y:", one_y)
print("three Y:", three_y)

<details>
<summary><b>Instructor solution / suggested answer</b></summary>


    A conventional Jordan–Wigner double-excitation generator produces eight nonzero Pauli strings: four with one \(Y\) and four with three \(Y\)'s, up to ordering/sign conventions.

    Suggested code:
    ```python
    one_y = sum(label.count("Y") == 1 for label in labels)
    three_y = sum(label.count("Y") == 3 for label in labels)
    ```

</details>